# 02. 기본 학습 실험 일지

## Step 0. 이 노트북을 작성하는 이유

`01_data_split.ipynb`에서는 Oxford-IIIT Pet Dataset을 dog/cat 이진 분류 문제로 바꾸고, train/validation/test split을 직접 만들었습니다.

이번 노트북에서는 저장해둔 split CSV를 불러와서 PyTorch로 기본 학습 파이프라인을 만듭니다. 목표는 높은 성능을 바로 내는 것이 아니라, 데이터가 모델에 들어가고 loss가 계산되고 optimizer가 가중치를 업데이트하는 흐름을 직접 설명할 수 있게 만드는 것입니다.

## Step 1. 이번 학습 실험의 목표를 정리합니다

이번 단계의 목표는 기본 데이터셋으로 baseline 모델을 학습하는 것입니다.

비교 선택지는 다음과 같습니다.

- **간단한 CNN 직접 구현:** 구조가 단순해서 학습 흐름을 설명하기 좋습니다.
- **pretrained ResNet 사용:** 성능은 좋을 수 있지만 처음부터 사용하면 직접 구현한 느낌이 약해질 수 있습니다.
- **더 깊은 CNN 직접 구현:** 실험은 가능하지만 초반에는 코드가 불필요하게 복잡해질 수 있습니다.

이번에는 **간단한 CNN 직접 구현**을 선택합니다. 과제에서 중요한 것은 완벽한 성능보다 접근 과정과 학습 흐름을 설명하는 것이기 때문입니다.

## Step 2. 필요한 라이브러리를 불러옵니다

이번 노트북에서는 CSV를 읽고, 이미지를 불러오고, PyTorch Dataset/DataLoader와 모델 학습 코드를 작성합니다.

In [ ]:
from pathlib import Path
import time

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
# 노트북을 어느 위치에서 실행해도 프로젝트 폴더를 찾을 수 있게 합니다.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

PROJECT_DIR

## Step 3. 01에서 저장한 split CSV를 불러옵니다

`01_data_split.ipynb`에서 train/validation/test 결과를 CSV로 저장했습니다. 여기서는 그 파일을 다시 읽어서 같은 데이터 분리를 그대로 사용합니다.

이렇게 하면 학습할 때마다 데이터가 새로 나뉘는 문제를 피할 수 있고, 나중에 실험 결과를 더 공정하게 비교할 수 있습니다.

In [ ]:
SPLIT_CSV_PATH = PROCESSED_DIR / "pet_binary_split_70_15_15_breed_stratified.csv"

split_df = pd.read_csv(SPLIT_CSV_PATH)
split_df.head()

## Step 4. 사용할 split 비율을 선택합니다

`01`에서는 80/10/10, 70/15/15, 60/20/20 세 가지 split을 만들었습니다.

이번 baseline에서는 **70/15/15 split**을 사용합니다. train 데이터가 충분히 많으면서 validation과 test도 너무 작지 않아, 처음 실험 기준으로 적당하다고 판단했습니다.

In [ ]:
train_df = split_df[split_df["split"] == "train"].reset_index(drop=True)
val_df = split_df[split_df["split"] == "validation"].reset_index(drop=True)
test_df = split_df[split_df["split"] == "test"].reset_index(drop=True)

print("train:", len(train_df))
print("validation:", len(val_df))
print("test:", len(test_df))

## Step 5. 이미지 전처리 방식을 결정합니다

이미지는 크기가 서로 다를 수 있기 때문에 모델에 넣기 전에 같은 크기로 맞춰야 합니다.

비교 선택지는 다음과 같습니다.

- **Resize만 사용:** 가장 단순하고 이해하기 쉽습니다.
- **Resize + Normalize 사용:** 딥러닝에서 자주 쓰는 기본 전처리입니다.
- **강한 augmentation 사용:** 성능 개선에 도움이 될 수 있지만 baseline 단계에서는 원인을 해석하기 어려워질 수 있습니다.

이번 baseline에서는 **Resize + Normalize**를 사용합니다. OpenCV 증강은 이후 발전 실험에서 따로 비교합니다.

In [ ]:
IMAGE_SIZE = 128

base_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

## Step 6. PyTorch Dataset을 만듭니다

PyTorch의 `Dataset`은 데이터 한 개를 어떻게 가져올지 정하는 역할을 합니다.

여기서는 CSV의 `image_path`를 보고 이미지를 열고, `label`을 숫자로 바꾼 뒤, 이미지와 라벨을 함께 반환합니다.

In [ ]:
LABEL_TO_INDEX = {"cat": 0, "dog": 1}
INDEX_TO_LABEL = {0: "cat", 1: "dog"}


class PetBinaryDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = Path(row["image_path"])
        if not image_path.is_absolute():
            image_path = PROJECT_DIR / image_path

        image = Image.open(image_path).convert("RGB")
        label = LABEL_TO_INDEX[row["label"]]

        if self.transform is not None:
            image = self.transform(image)

        return image, label

## Step 7. DataLoader를 만듭니다

`DataLoader`는 Dataset에서 데이터를 여러 장씩 묶어서 모델에 전달합니다.

train set은 학습 순서에 모델이 익숙해지지 않도록 `shuffle=True`를 사용합니다. validation과 test는 평가용이므로 섞지 않아도 됩니다.

In [ ]:
BATCH_SIZE = 32

train_dataset = PetBinaryDataset(train_df, transform=base_transform)
val_dataset = PetBinaryDataset(val_df, transform=base_transform)
test_dataset = PetBinaryDataset(test_df, transform=base_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

sample_images, sample_labels = next(iter(train_loader))
print(sample_images.shape)
print(sample_labels[:10])

## Step 8. baseline 모델을 선택합니다

이번에는 간단한 CNN을 직접 만듭니다.

CNN은 이미지에서 작은 패턴을 찾는 `Conv2d`, 크기를 줄이는 `MaxPool2d`, 최종 분류를 담당하는 `Linear` layer로 구성됩니다. 처음부터 복잡한 모델을 쓰기보다, 구조를 설명할 수 있는 모델로 baseline을 만드는 것이 목적입니다.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## Step 9. loss function과 optimizer를 선택합니다

loss function은 모델의 예측이 정답과 얼마나 다른지 계산합니다. optimizer는 loss를 줄이는 방향으로 모델의 가중치를 업데이트합니다.

이번 문제는 cat/dog 두 클래스 중 하나를 고르는 분류 문제이므로 `CrossEntropyLoss`를 사용합니다. optimizer는 처음 baseline에서 많이 사용하는 `Adam`을 선택합니다.

In [ ]:
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model

## Step 10. CPU/GPU device를 선택합니다

PyTorch에서는 모델과 데이터를 같은 device에 올려야 합니다. GPU를 사용할 수 있으면 `cuda`, 아니면 `cpu`를 사용합니다.

현재 설치된 PyTorch가 CPU 전용이면 컴퓨터에 GPU가 있어도 `torch.cuda.is_available()`이 `False`가 될 수 있습니다. 이 경우 나중에 CUDA용 PyTorch 설치를 따로 확인해야 합니다.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("device:", device)
print("cuda available:", torch.cuda.is_available())

## Step 11. train 1 epoch 함수를 만듭니다

train 함수는 모델이 실제로 공부하는 단계입니다.

과제 요구사항에서 중요한 흐름은 다음 순서입니다.

`optimizer.zero_grad -> forward -> loss -> backward -> step`

이 흐름이 직접 코드에 보이도록 작성합니다.

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # 1. 이전 batch에서 계산된 gradient를 지웁니다.
        optimizer.zero_grad()

        # 2. 모델이 이미지를 보고 cat/dog 점수를 예측합니다. 이 단계가 forward입니다.
        outputs = model(images)

        # 3. 예측값과 정답 라벨을 비교해서 오차(loss)를 계산합니다.
        loss = criterion(outputs, labels)

        # 4. loss를 줄이려면 각 가중치를 어느 방향으로 바꿔야 하는지 계산합니다.
        loss.backward()

        # 5. 계산된 gradient를 이용해 모델의 가중치를 실제로 업데이트합니다.
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += batch_size

    average_loss = total_loss / total
    accuracy = correct / total
    return average_loss, accuracy

## Step 12. validation 함수를 만듭니다

validation은 모델을 업데이트하지 않고 현재 실력을 확인하는 단계입니다.

그래서 `model.eval()`과 `torch.no_grad()`를 사용합니다. 이 단계에서는 `loss.backward()`와 `optimizer.step()`을 하지 않습니다.

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += batch_size

    average_loss = total_loss / total
    accuracy = correct / total
    return average_loss, accuracy

## Step 13. 전체 학습 루프를 만듭니다

전체 학습 루프는 train과 validation을 epoch마다 반복합니다.

처음에는 오래 걸리지 않게 `NUM_EPOCHS = 3`으로 설정합니다. 나중에 시간이 괜찮으면 epoch 수를 늘려서 loss curve가 어떻게 변하는지 볼 수 있습니다.

In [ ]:
NUM_EPOCHS = 3

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"train loss: {train_loss:.4f}, train acc: {train_accuracy:.4f} | "
        f"val loss: {val_loss:.4f}, val acc: {val_accuracy:.4f}"
    )

elapsed_time = time.time() - start_time
print(f"elapsed time: {elapsed_time:.2f} seconds")

## Step 14. train/validation loss curve를 그립니다

loss curve를 보면 학습이 진행되면서 train loss와 validation loss가 어떻게 변하는지 확인할 수 있습니다.

나중에 train loss는 계속 내려가는데 validation loss가 올라간다면 overfitting을 의심할 수 있습니다.

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], marker="o", label="train loss")
plt.plot(epochs, history["val_loss"], marker="o", label="validation loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Baseline Train/Validation Loss")
plt.legend()
plt.tight_layout()

loss_curve_path = FIGURE_DIR / "baseline_loss_curve.png"
plt.savefig(loss_curve_path, dpi=150)
plt.show()

print(loss_curve_path)

## Step 15. CPU/GPU 학습 시간 비교를 위한 기록을 남깁니다

나중에 같은 조건으로 CPU와 GPU에서 각각 학습해보고 시간을 비교합니다.

비교할 때는 모델, split, batch size, epoch 수를 같게 두고 device만 바꾸는 것이 좋습니다. 그래야 시간 차이가 CPU/GPU 차이에서 온 것인지 해석하기 쉽습니다.

In [ ]:
training_record = {
    "split_csv": str(SPLIT_CSV_PATH),
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": NUM_EPOCHS,
    "device": str(device),
    "elapsed_time_seconds": elapsed_time,
    "final_train_loss": history["train_loss"][-1],
    "final_val_loss": history["val_loss"][-1],
    "final_train_accuracy": history["train_accuracy"][-1],
    "final_val_accuracy": history["val_accuracy"][-1],
}

training_record

## Step 16. 결과를 보고 다음 실험 계획을 정리합니다

먼저 기본 데이터셋으로 baseline을 만들고, 이후 OpenCV 증강 데이터셋과 비교하는 방식으로 접근했습니다.

다음 단계에서는 loss curve를 보고 overfitting 여부를 판단하고, accuracy뿐 아니라 confusion matrix, precision, recall을 계산합니다. 또한 OpenCV로 조명, 대비, 회전, 흐림 등을 적용한 train 증강 데이터셋을 만들어 기본 데이터셋과 비교할 계획입니다.

중요한 점은 증강을 split 전에 적용하지 않는 것입니다. 증강은 train set에만 적용하고 validation/test는 원본 기준으로 유지해야 공정한 비교가 가능합니다.